# Full Two-Gap Model Charts

Loads the empirical two-gap model, reconstructs efficient and cost-push components, plots trends and posterior diagnostics, and compares the output gaps with benchmark estimates.

## Initialization

In [1]:
using CSV, Dates, DataFrames, Distributions, FileIO, JLD2, KernelDensity, LinearAlgebra, PlotlyJS, PlotlyKaleido, Statistics
PlotlyKaleido.start()
pltjs = PlotlyJS

include("./code/Metropolis-Within-Gibbs/MetropolisWithinGibbs.jl");
using Main.MetropolisWithinGibbs;
include("./code/filters.jl");


┌ Warning: Kaledio is not available on this system. Julia will be unable to produce any plots.
└ @ PlotlyBase C:\Users\wrc938\.julia\packages\PlotlyBase\NxSlF\src\kaleido.jl:58


## Helpers

In [2]:
SIGMA_Y_KEY = String([Char(0x03c3), Char(0x02b8)])
ALPHA_KEY = "distr_" * String([Char(0x03b1)])
THETA_BOUND_KEY = "chain_" * String([Char(0x03b8)]) * "_bound"
LAMBDA_FIELD = Symbol(String([Char(0x03bb)]))
RHO_FIELD = Symbol(String([Char(0x03c1)]))

function require_file(path; hint="")
    if !isfile(path)
        msg = isempty(hint) ? "Missing required file: $path" : "Missing required file: $path. $hint"
        error(msg)
    end
    return path
end

function first_existing(paths)
    for p in paths
        if isfile(p)
            return p
        end
    end
    return nothing
end

function par_size_from(par_ind)
    n_d = sum(par_ind.d); n_Z = sum(par_ind.Z); n_Z_plus = sum(par_ind.Z_plus); n_Z_minus = sum(par_ind.Z_minus)
    n_R = sum(par_ind.R); n_c = sum(par_ind.c); n_T = sum(par_ind.T); n_Q = sum(par_ind.Q); n_Q_cov = sum(par_ind.Q_cov)
    n_lambda = sum(getfield(par_ind, LAMBDA_FIELD)); n_rho = sum(getfield(par_ind, RHO_FIELD))
    return SizeParSsm(n_d, n_Z, n_Z_plus, n_Z_minus, n_R, n_c, n_T, n_Q, n_Q_cov,
                      n_lambda, n_rho, n_d + n_Z + n_Z_plus + n_Z_minus + n_R + n_c + n_T + n_Q + n_Q_cov + n_lambda + n_rho)
end

function as_float_matrix(x)
    out = Array{Float64}(undef, size(x))
    for i in eachindex(x)
        out[i] = ismissing(x[i]) ? NaN : Float64(x[i])
    end
    return out
end

function as_float_vector(x)
    out = Vector{Float64}(undef, length(x))
    for i in eachindex(x)
        out[i] = ismissing(x[i]) ? NaN : Float64(x[i])
    end
    return out
end

function extend_quarterly_dates(date, target_len)
    out = DateTime.(date)
    while length(out) < target_len
        m = Dates.month(out[end]) + 3
        y = Dates.year(out[end])
        if m > 12
            y += 1
            m -= 12
        end
        push!(out, DateTime(Dates.lastdayofquarter(Date(y, m, 1))))
    end
    return out
end

function summarize_component(draws3, scales)
    n, TT, _ = size(draws3)
    med = zeros(TT, n); lo90 = zeros(TT, n); hi90 = zeros(TT, n); lo68 = zeros(TT, n); hi68 = zeros(TT, n)
    for i in 1:n, t in 1:TT
        xi = filter(isfinite, vec(draws3[i, t, :]) .* scales[i])
        if isempty(xi)
            med[t, i] = NaN; lo90[t, i] = NaN; hi90[t, i] = NaN; lo68[t, i] = NaN; hi68[t, i] = NaN
        else
            med[t, i] = median(xi)
            lo90[t, i] = quantile(xi, 0.05)
            hi90[t, i] = quantile(xi, 0.95)
            lo68[t, i] = quantile(xi, 0.16)
            hi68[t, i] = quantile(xi, 0.84)
        end
    end
    return (; med, lo90, hi90, lo68, hi68)
end

function summarize_matrix(draws2)
    TT, _ = size(draws2)
    med = zeros(TT); lo90 = zeros(TT); hi90 = zeros(TT); lo68 = zeros(TT); hi68 = zeros(TT)
    for t in 1:TT
        xi = filter(isfinite, vec(draws2[t, :]))
        if isempty(xi)
            med[t] = NaN; lo90[t] = NaN; hi90[t] = NaN; lo68[t] = NaN; hi68[t] = NaN
        else
            med[t] = median(xi); lo90[t] = quantile(xi, 0.05); hi90[t] = quantile(xi, 0.95)
            lo68[t] = quantile(xi, 0.16); hi68[t] = quantile(xi, 0.84)
        end
    end
    return (; med, lo90, hi90, lo68, hi68)
end

function band_traces(x, s; col="rgba(42,110,166,1)", fill90="rgba(42,110,166,0.15)", fill68="rgba(42,110,166,0.30)", name="Median", show_ci=false, dash=nothing)
    line_attr = dash === nothing ? attr(color=col, width=2.5) : attr(color=col, width=2.5, dash=dash)
    return [
        pltjs.scatter(x=x, y=s.lo90, mode="lines", line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
        pltjs.scatter(x=x, y=s.hi90, mode="lines", fill="tonexty", fillcolor=fill90, line=attr(color="rgba(0,0,0,0)"), name="90% CI", showlegend=show_ci),
        pltjs.scatter(x=x, y=s.lo68, mode="lines", line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
        pltjs.scatter(x=x, y=s.hi68, mode="lines", fill="tonexty", fillcolor=fill68, line=attr(color="rgba(0,0,0,0)"), name="68% CI", showlegend=show_ci),
        pltjs.scatter(x=x, y=s.med, mode="lines", name=name, line=line_attr)
    ]
end

base_layout(title; ylabel="Pct.") = pltjs.Layout(template="plotly_white", title=title, hovermode="x unified",
    width=900, height=420, margin=attr(l=60, r=20, t=60, b=55),
    xaxis=attr(title="Time", showgrid=true, gridcolor="rgba(0,0,0,0.08)", zeroline=false, showline=true, linecolor="black", mirror=true),
    yaxis=attr(title=ylabel, showgrid=true, gridcolor="rgba(0,0,0,0.08)", zeroline=true, showline=true, linecolor="black", mirror=true),
    legend=attr(orientation="h", x=0.0, y=1.02, yanchor="bottom", bgcolor="rgba(0,0,0,0)"))

function save_and_display(fig, filename)
    path = joinpath(results_folder, filename)
    PlotlyKaleido.savefig(fig, path)
    println("Saved figure: ", path)
    return path
end

function finite_cor(x, y)
    valid = isfinite.(x) .& isfinite.(y)
    return cor(x[valid], y[valid])
end

function finite_rel_vol(x, y)
    valid = isfinite.(x) .& isfinite.(y)
    return std(x[valid]) / std(y[valid])
end


finite_rel_vol (generic function with 1 method)

## Load Two-Gap Results

In [3]:
res = load(require_file("./res_two_gap_iis.jld2"; hint="Run user_main.jl with the current two-gap IIS specification first."))

results_folder = "results_two_gap_iis"
mkpath(results_folder)

nDraws = res["nDraws"]
burnin = res["burnin"]
sigma_y = vec(res[SIGMA_Y_KEY])
date0 = res["date"]
data = as_float_matrix(res["data"] .* reshape(sigma_y, 1, :))
alpha = res[ALPHA_KEY]
chain = res[THETA_BOUND_KEY]
distr_par = res["distr_par"]
par_ind = res["par_ind"]
par_size = par_size_from(par_ind)
MNEMONIC = res["MNEMONIC"]

titles = ["Real GDP", "Employment", "Unemployment rate", "Inflation rate", "UoM expected inflation", "SPF expected inflation"]
scales = ["Pct.", "Pct.", "Pct.", "Pct.", "Pct.", "Pct."]

n = size(data, 2)
TT = size(alpha, 2)
n_smpl = size(alpha, 3)
post = size(chain, 2) == nDraws[2] ? collect(burnin[2] + 1:nDraws[2]) : collect(1:size(chain, 2))
@assert length(post) >= n_smpl "Mismatch between parameter draws and state draws"
post = post[1:n_smpl]
chain_post = chain[:, post]
date = extend_quarterly_dates(date0, TT)
max_h = TT - length(date0)
hist = 1:(TT - max_h)
x = date[hist]

println("Observables: ", MNEMONIC)
println("State draws: ", size(alpha))
println("Parameter size: ", par_size)
println("Saving figures to: ", results_folder)


Observables: ["GDP", "EMPL", "U", "CORE", "UOM", "SPF"]
State draws: (22, 174, 20000)
Parameter size: SizeParSsm{Int64}(0, 2, 3, 0, 0, 2, 0, 14, 1, 8, 8, 38)
Saving figures to: results_two_gap_iis


## Reconstruct Model Components

In [4]:
# The current repository's two-gap model is AR(2). The stored Z matrices contain the RE-implied
# loadings for each posterior draw, so use them instead of hard-coded one-state loadings.
ind_cycles = [6, 9, 12, 15, 17, 20]
ind_trends = [8, 11, 14, 19, 22]
trend_obs_idx = [1, 2, 3, 5, 6]

Z_matrices = [distr_par[k].Z for k in 1:n_smpl]
sigma_scale = reshape(sigma_y, n, 1)

efficient = zeros(n, TT, n_smpl)
cost_push = zeros(n, TT, n_smpl)
infl_trend = zeros(n, TT, n_smpl)

for d in 1:n_smpl
    Z = Z_matrices[d]
    efficient[:, :, d] = (Z[:, 1] .* alpha[1, :, d]') .+ (Z[:, 2] .* alpha[2, :, d]')
    cost_push[:, :, d] = (Z[:, 3] .* alpha[3, :, d]') .+ (Z[:, 4] .* alpha[4, :, d]')
    infl_trend[:, :, d] = Z[:, 5] .* alpha[5, :, d]'

    efficient[:, :, d] .*= sigma_scale
    cost_push[:, :, d] .*= sigma_scale
    infl_trend[:, :, d] .*= sigma_scale
end

idx_kappa = par_size.R + par_size.d + par_size.Z + 1
kappa = chain_post[idx_kappa, :]
flex_gap_draws = zeros(TT, n_smpl)
flex_potential_draws = fill(NaN, TT, n_smpl)

for d in 1:n_smpl
    flex_gap_draws[:, d] = alpha[1, :, d] .+ alpha[3, :, d] ./ kappa[d]
    for t in hist
        flex_potential_draws[t, d] = data[t, 1] - flex_gap_draws[t, d]
    end
end

idio_cycle = alpha[ind_cycles, :, :]
idio_trend = alpha[ind_trends, :, :]

eff = summarize_component(efficient, ones(n))
cost = summarize_component(cost_push, ones(n))
trend = summarize_component(infl_trend, ones(n))
iC = summarize_component(idio_cycle, sigma_y)
iT = summarize_component(idio_trend, sigma_y[trend_obs_idx])
flex_gap = summarize_matrix(flex_gap_draws)
flex_potential = summarize_matrix(flex_potential_draws)

println("Efficient component size: ", size(efficient))
println("Cost-push component size: ", size(cost_push))
println("Posterior median of kappa: ", median(kappa))


Efficient component size: (6, 174, 20000)
Cost-push component size: (6, 174, 20000)
Posterior median of kappa: 0.28045535292201945


## Historical Decompositions

In [5]:
function decomposition_fig(obs_range; filename)
    figs = Any[]
    for i in obs_range
        show_legend = i == first(obs_range)
        traces = [
            pltjs.bar(x=x, y=eff.med[hist, i], name="Efficient gap", marker_color="rgba(0,48,158,0.75)", showlegend=show_legend),
            pltjs.bar(x=x, y=cost.med[hist, i], name="Cost-push cycle", marker_color="rgba(255,0,0,0.65)", showlegend=show_legend),
            pltjs.bar(x=x, y=iC.med[hist, i], name="Idiosyncratic cycle", marker_color="rgba(255,190,0,0.75)", showlegend=show_legend),
            pltjs.scatter(x=x, y=eff.med[hist, i] .+ cost.med[hist, i] .+ iC.med[hist, i],
                          mode="lines", name="Total cycle", line=attr(width=1.4, color="black"), showlegend=show_legend)
        ]
        layout = pltjs.Layout(title=titles[i], titlefont_size=12,
            xaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, titlefont=attr(size=10), nticks=20, tickangle=-90),
            yaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, titlefont=attr(size=10), title=scales[i]),
            barmode="relative", bargap=0.02, bargroupgap=0,
            margin=attr(l=45, r=25, t=45, b=45), legend=attr(orientation="h", y=-0.25, x=0.05))
        push!(figs, pltjs.plot(traces, layout))
    end

    fig = [figs[1]; figs[2]; figs[3]]
    fig.plot.layout["barmode"] = "relative"
    fig.plot.layout["bargap"] = 0.02
    fig.plot.layout["width"] = 800
    fig.plot.layout["height"] = 600
    fig.plot.layout["margin"][:b] = 40
    fig.plot.layout["margin"][:t] = 40
    fig.plot.layout["margin"][:r] = 40
    fig.plot.layout["margin"][:l] = 40
    fig.plot.layout["legend"] = attr(orientation="h", y=-0.1, x=0.075, font=attr(size=10))
    try
        for ann in fig.plot.layout["annotations"]
            ann[:font][:size] = 10
        end
    catch
    end
    save_and_display(fig, filename)
    return fig
end

decomp_1_3 = decomposition_fig(1:3; filename="historical_decomposition_observables_1_3.png")
decomp_4_6 = decomposition_fig(4:6; filename="historical_decomposition_observables_4_5.png")

nothing


Saved figure: results_two_gap_iis\historical_decomposition_observables_1_3.png
Saved figure: results_two_gap_iis\historical_decomposition_observables_4_5.png


## Posterior Diagnostics

In [15]:
function density_plot(draws, prior_x, prior_y; title="Posterior", x_range=nothing, showlegend=false)
    k = kde(vec(draws))

    traces = [
        pltjs.scatter(x=prior_x, y=prior_y, mode="lines", name="Prior", legendgroup="prior", showlegend=showlegend,
                      line=attr(color="rgba(255,0,0,0.75)", width=1.4), hoverinfo="skip"),
        pltjs.scatter(x=k.x, y=k.density, mode="lines", name="Posterior", legendgroup="posterior", showlegend=showlegend,
                      line=attr(color="rgba(0,48,158,0.75)", width=1.4), hoverinfo="skip")
    ]

    xaxis = x_range === nothing ?
        attr(showgrid=true, linecolor="black", mirror=true) :
        attr(showgrid=true, linecolor="black", mirror=true, range=x_range)

    return pltjs.plot(traces, pltjs.Layout(title=title, titlefont_size=12, width=350, height=260,
        template="plotly_white", xaxis=xaxis, yaxis=attr(showgrid=true, linecolor="black", mirror=true), showlegend=true))
end

idx_Q = par_size.R + par_size.d + par_size.Z + par_size.Z_plus + par_size.Z_minus
idx_Q_cov = idx_Q + par_size.Q
idx_lambda = idx_Q_cov + par_size.Q_cov + par_size.c + par_size.T
idx_rho = idx_lambda + getfield(par_size, LAMBDA_FIELD)

x_freq = collect(0.001:0.01:pi)
x_rho = collect(0.001:0.01:0.97)
x_var = collect(0.001:0.01:1.5)
x_cov = collect(-0.97:0.01:0.97)
x_pc = collect(-1.0:0.02:1.0)

fig_freq_eff = density_plot(chain_post[idx_lambda + 1, :], x_freq, pdf.(Uniform(0.001, pi), x_freq);
    title="Frequency, efficient cycle", x_range=[0, pi], showlegend=true)
fig_freq_cost = density_plot(chain_post[idx_lambda + 2, :], x_freq, pdf.(Uniform(0.001, pi), x_freq);
    title="Frequency, cost-push cycle", x_range=[0, pi])
fig_rho_eff = density_plot(chain_post[idx_rho + 1, :], x_rho, pdf.(Uniform(0.001, 0.97), x_rho);
    title="Persistence, efficient cycle", x_range=[0, 0.97])
fig_rho_cost = density_plot(chain_post[idx_rho + 2, :], x_rho, pdf.(Uniform(0.001, 0.97), x_rho);
    title="Persistence, cost-push cycle", x_range=[0, 0.97])
fig_var_eff = density_plot(chain_post[idx_Q + 1, :], x_var, pdf.(InverseGamma(3, 1), x_var);
    title="Variance, efficient cycle", x_range=[0, 1.5])
fig_var_cost = density_plot(chain_post[idx_Q + 3, :], x_var, pdf.(InverseGamma(3, 1), x_var);
    title="Variance, cost-push cycle", x_range=[0, 1.5])
fig_covar = par_size.Q_cov > 0 ? density_plot(chain_post[idx_Q_cov + 1, :], x_cov, pdf.(Uniform(-0.97, 0.97), x_cov);
    title="Correlation, cycles", x_range=[-0.97, 0.97]) : pltjs.plot()
fig_phillips = density_plot(chain_post[idx_kappa, :], x_pc, pdf.(Normal(0, 1000), x_pc);
    title="Phillips curve slope", x_range=[-1, 1])

posterior_fig = [fig_freq_eff fig_rho_eff fig_var_eff fig_phillips; fig_freq_cost fig_rho_cost fig_var_cost fig_covar]
posterior_fig.plot.layout["width"] = 1000
posterior_fig.plot.layout["height"] = 600
posterior_fig.plot.layout["margin"][:b] = 40
posterior_fig.plot.layout["margin"][:t] = 40
posterior_fig.plot.layout["margin"][:r] = 40
posterior_fig.plot.layout["margin"][:l] = 40
posterior_fig.plot.layout["legend"] = attr(orientation="h", y=-0.1, x=0.42, font=attr(size=10))
try
    for ann in posterior_fig.plot.layout["annotations"]
        ann[:font][:size] = 10
    end
catch
end
save_and_display(posterior_fig, "posterior_key_parameters_combined.png")

println("Median Phillips curve slope: ", median(chain_post[idx_kappa, :]))

nothing


Saved figure: results_two_gap_iis\posterior_key_parameters_combined.png
Median Phillips curve slope: 0.28045535292201945


## Gap and Benchmark Comparisons

In [7]:
eff_gap = (; med=eff.med[hist, 1], lo90=eff.lo90[hist, 1], hi90=eff.hi90[hist, 1], lo68=eff.lo68[hist, 1], hi68=eff.hi68[hist, 1])
cost_infl = (; med=cost.med[hist, 4], lo90=cost.lo90[hist, 4], hi90=cost.hi90[hist, 4], lo68=cost.lo68[hist, 4], hi68=cost.hi68[hist, 4])
flex_hist = (; med=flex_gap.med[hist], lo90=flex_gap.lo90[hist], hi90=flex_gap.hi90[hist], lo68=flex_gap.lo68[hist], hi68=flex_gap.hi68[hist])

gap_fig = pltjs.plot([
    band_traces(x, flex_hist; col="#B33A3A", fill90="rgba(179,58,58,0.15)", fill68="rgba(179,58,58,0.30)", name="Flexible-price gap")...,
    band_traces(x, eff_gap; col="#2A6EA6", fill90="rgba(42,110,166,0.15)", fill68="rgba(42,110,166,0.30)", name="Efficient gap")...,
    band_traces(x, cost_infl; col="#3D9970", fill90="rgba(61,153,112,0.15)", fill68="rgba(61,153,112,0.30)", name="Cost-push cycle")...
], base_layout("Two-gap estimates"))
save_and_display(gap_fig, "gaps_efficient_cost_push_and_flex.png")

cbo_gap = as_float_vector(CSV.read(require_file("./data/CBO_gap_2025.csv"; hint="Expected column GDPC1_GDPPOT."), DataFrame)[!, :GDPC1_GDPPOT])
hp_gap = HP_filter(data[hist, 1])
L = minimum([length(x), length(cbo_gap), length(hp_gap)])

bench_fig = pltjs.plot([
    band_traces(x[1:L], (; med=flex_hist.med[1:L], lo90=flex_hist.lo90[1:L], hi90=flex_hist.hi90[1:L], lo68=flex_hist.lo68[1:L], hi68=flex_hist.hi68[1:L]); col="#B33A3A", fill90="rgba(179,58,58,0.12)", fill68="rgba(179,58,58,0.24)", name="Flexible-price gap")...,
    band_traces(x[1:L], (; med=eff_gap.med[1:L], lo90=eff_gap.lo90[1:L], hi90=eff_gap.hi90[1:L], lo68=eff_gap.lo68[1:L], hi68=eff_gap.hi68[1:L]); col="#2A6EA6", fill90="rgba(42,110,166,0.12)", fill68="rgba(42,110,166,0.24)", name="Efficient gap")...,
    pltjs.scatter(x=x[1:L], y=hp_gap[1:L], mode="lines", name="HP gap", line=attr(color="black", width=2.0, dash="dash")),
    pltjs.scatter(x=x[1:L], y=cbo_gap[1:L], mode="lines", name="CBO gap", line=attr(color="black", width=2.0, dash="dot"))
], base_layout("Two-gap output gaps with HP and CBO benchmarks"))
save_and_display(bench_fig, "gaps_with_HP_CBO_benchmarks.png")

benchmark_summary = DataFrame(
    comparison=["Efficient vs HP", "Flexible vs HP", "Efficient vs CBO", "Flexible vs CBO"],
    correlation=[finite_cor(eff_gap.med[1:L], hp_gap[1:L]), finite_cor(flex_hist.med[1:L], hp_gap[1:L]), finite_cor(eff_gap.med[1:L], cbo_gap[1:L]), finite_cor(flex_hist.med[1:L], cbo_gap[1:L])],
    relative_volatility=[finite_rel_vol(hp_gap[1:L], eff_gap.med[1:L]), finite_rel_vol(hp_gap[1:L], flex_hist.med[1:L]), finite_rel_vol(cbo_gap[1:L], eff_gap.med[1:L]), finite_rel_vol(cbo_gap[1:L], flex_hist.med[1:L])]
)
display(benchmark_summary)

nothing


Saved figure: results_two_gap_iis\gaps_efficient_cost_push_and_flex.png
Saved figure: results_two_gap_iis\gaps_with_HP_CBO_benchmarks.png


Row,comparison,correlation,relative_volatility
,String,Float64,Float64
1,Efficient vs HP,0.686279,0.725569
2,Flexible vs HP,0.432791,0.390082
3,Efficient vs CBO,0.777079,1.08073
4,Flexible vs CBO,0.49979,0.581025


## Comparator Model Overlays

In [8]:
function optional_one_gap_summary(path)
    if !isfile(path)
        println("Skipping optional comparator; missing ", path)
        return nothing
    end
    r = load(path)
    sigma = vec(r[SIGMA_Y_KEY])
    a = r[ALPHA_KEY]
    return summarize_matrix(a[1, :, :] .* sigma[1])
end

kuttner_gap = optional_one_gap_summary("./res_kuttner_iis.jld2")
okun_gap = optional_one_gap_summary("./res_okun_kuttner_iis.jld2")

overlay_traces = [
    band_traces(x, flex_hist; col="#B33A3A", fill90="rgba(179,58,58,0.10)", fill68="rgba(179,58,58,0.20)", name="Flexible-price gap")...,
    band_traces(x, eff_gap; col="#2A6EA6", fill90="rgba(42,110,166,0.10)", fill68="rgba(42,110,166,0.20)", name="Efficient gap")...
]
if kuttner_gap !== nothing
    Lk = min(length(x), length(kuttner_gap.med))
    kuttner_hist = (; med=kuttner_gap.med[1:Lk], lo90=kuttner_gap.lo90[1:Lk], hi90=kuttner_gap.hi90[1:Lk], lo68=kuttner_gap.lo68[1:Lk], hi68=kuttner_gap.hi68[1:Lk])
    append!(overlay_traces, band_traces(x[1:Lk], kuttner_hist; col="#111111", fill90="rgba(0,0,0,0.08)", fill68="rgba(0,0,0,0.16)", name="Kuttner gap", dash="dot"))
end
if okun_gap !== nothing
    Lo = min(length(x), length(okun_gap.med))
    okun_hist = (; med=okun_gap.med[1:Lo], lo90=okun_gap.lo90[1:Lo], hi90=okun_gap.hi90[1:Lo], lo68=okun_gap.lo68[1:Lo], hi68=okun_gap.hi68[1:Lo])
    append!(overlay_traces, band_traces(x[1:Lo], okun_hist; col="#555555", fill90="rgba(80,80,80,0.08)", fill68="rgba(80,80,80,0.16)", name="Okun-Kuttner gap", dash="dash"))
end
overlay_fig = pltjs.plot(overlay_traces, base_layout("Full model vs one-gap comparators"))
save_and_display(overlay_fig, "gaps_comparison_with_Kuttner_Okun.png")

nothing


Saved figure: results_two_gap_iis\gaps_comparison_with_Kuttner_Okun.png


## Trends and CBO Potential / NAIRU

In [9]:
cbo_potential = as_float_vector(CSV.read(require_file("./data/CBO_potential_2025.csv"; hint="Expected column GDPPOT."), DataFrame)[!, :GDPPOT])
cbo_nairu = as_float_vector(CSV.read(require_file("./data/CBO_NROU_2025.csv"; hint="Expected column NROU."), DataFrame)[!, :NROU])
cbo_potential = 100 .* log.(cbo_potential)

Lp = minimum([length(x), length(cbo_potential), length(flex_potential.med), size(iT.med, 1)])
potential_fig = pltjs.plot([
    band_traces(x[1:Lp], (; med=flex_potential.med[1:Lp], lo90=flex_potential.lo90[1:Lp], hi90=flex_potential.hi90[1:Lp], lo68=flex_potential.lo68[1:Lp], hi68=flex_potential.hi68[1:Lp]); col="#B33A3A", fill90="rgba(179,58,58,0.12)", fill68="rgba(179,58,58,0.24)", name="Flexible-price potential")...,
    band_traces(x[1:Lp], (; med=iT.med[1:Lp, 1], lo90=iT.lo90[1:Lp, 1], hi90=iT.hi90[1:Lp, 1], lo68=iT.lo68[1:Lp, 1], hi68=iT.hi68[1:Lp, 1]); col="#2A6EA6", fill90="rgba(42,110,166,0.12)", fill68="rgba(42,110,166,0.24)", name="Efficient potential")...,
    pltjs.scatter(x=x[1:Lp], y=cbo_potential[1:Lp], mode="lines", name="CBO potential", line=attr(color="black", width=2.2, dash="dash")),
    pltjs.scatter(x=x[1:Lp], y=data[1:Lp, 1], mode="lines", name="Actual output", line=attr(color="black", width=2.0, dash="dot"))
], base_layout("Potential output", ylabel="log(Y)*100"))
save_and_display(potential_fig, "potential_output_vs_CBO.png")

Lu = minimum([length(x), length(cbo_nairu), size(iT.med, 1)])
nairu_fig = pltjs.plot([
    band_traces(x[1:Lu], (; med=iT.med[1:Lu, 3], lo90=iT.lo90[1:Lu, 3], hi90=iT.hi90[1:Lu, 3], lo68=iT.lo68[1:Lu, 3], hi68=iT.hi68[1:Lu, 3]); col="#2A6EA6", fill90="rgba(42,110,166,0.12)", fill68="rgba(42,110,166,0.24)", name="Trend unemployment")...,
    pltjs.scatter(x=x[1:Lu], y=cbo_nairu[1:Lu], mode="lines", name="CBO NAIRU", line=attr(color="black", width=2.2, dash="dash")),
    pltjs.scatter(x=x[1:Lu], y=data[1:Lu, 3], mode="lines", name="Actual unemployment", line=attr(color="black", width=2.0, dash="dot"))
], base_layout("Trend unemployment and CBO NAIRU"))
save_and_display(nairu_fig, "trend_unemployment_vs_CBO_NAIRU.png")

infl_fig = pltjs.plot([
    band_traces(x, (; med=trend.med[hist, 4], lo90=trend.lo90[hist, 4], hi90=trend.hi90[hist, 4], lo68=trend.lo68[hist, 4], hi68=trend.hi68[hist, 4]); col="#2A6EA6", fill90="rgba(42,110,166,0.12)", fill68="rgba(42,110,166,0.24)", name="Inflation trend")...,
    pltjs.scatter(x=x, y=data[hist, 4], mode="lines", name="Observed CPI inflation", line=attr(color="black", width=2.0, dash="dot"))
], base_layout("Inflation trend"))
save_and_display(infl_fig, "inflation_trend_vs_actual.png")

nothing


Saved figure: results_two_gap_iis\potential_output_vs_CBO.png
Saved figure: results_two_gap_iis\trend_unemployment_vs_CBO_NAIRU.png
Saved figure: results_two_gap_iis\inflation_trend_vs_actual.png


## HZ Comparison

In [10]:
hz_path = first_existing(["./res_iis_HZ.jld2", "./res_hasenzagl_2020_iis.jld2"])
if hz_path === nothing
    println("Skipping HZ comparison: no HZ result file found among res_iis_HZ.jld2 or res_hasenzagl_2020_iis.jld2.")
else
    res_HZ = load(hz_path)
    nDraws_HZ = res_HZ["nDraws"]
    burnin_HZ = res_HZ["burnin"]
    sigma_HZ = vec(res_HZ[SIGMA_Y_KEY])
    date_HZ = extend_quarterly_dates(res_HZ["date"], size(res_HZ[ALPHA_KEY], 2))
    alpha_HZ = res_HZ[ALPHA_KEY]
    chain_HZ = res_HZ[THETA_BOUND_KEY]

    n_HZ = size(res_HZ["data"], 2)
    TT_HZ = size(alpha_HZ, 2)
    ndraw_HZ = nDraws_HZ[2] - burnin_HZ[2]
    post_HZ = burnin_HZ[2] + 1:nDraws_HZ[2]

    if n_HZ < 5 || size(alpha_HZ, 1) < 5 || size(chain_HZ, 1) < 24
        println("Skipping HZ energy-price-cycle comparison: ", hz_path, " does not look like the original HZ result.")
    else
        Z_energy = [zeros(3, ndraw_HZ); ones(1, ndraw_HZ); chain_HZ[17:20, post_HZ]]
        Z_energy_lag = [zeros(4, ndraw_HZ); chain_HZ[21:24, post_HZ]]
        EP_HZ = zeros(n_HZ, TT_HZ, ndraw_HZ)

        for d in 1:ndraw_HZ
            EP_HZ[:, :, d] = (Z_energy[:, d] .* alpha_HZ[4, :, d]') .+ (Z_energy_lag[:, d] .* alpha_HZ[5, :, d]')
        end

        ep_hz = summarize_component(EP_HZ, sigma_HZ)
        Lcp = minimum([length(x), length(date_HZ), size(cost.med, 1), size(ep_hz.med, 1)])
        cost_push_cpi = (; med=cost.med[1:Lcp, 4], lo90=cost.lo90[1:Lcp, 4], hi90=cost.hi90[1:Lcp, 4], lo68=cost.lo68[1:Lcp, 4], hi68=cost.hi68[1:Lcp, 4])
        energy_cycle_hz = (; med=ep_hz.med[1:Lcp, 5], lo90=ep_hz.lo90[1:Lcp, 5], hi90=ep_hz.hi90[1:Lcp, 5], lo68=ep_hz.lo68[1:Lcp, 5], hi68=ep_hz.hi68[1:Lcp, 5])

        cp_ep_fig = pltjs.plot([
            band_traces(x[1:Lcp], cost_push_cpi; col="#3D9970", fill90="rgba(61,153,112,0.15)", fill68="rgba(61,153,112,0.30)", name="Cost-push cycle")...,
            band_traces(date_HZ[1:Lcp], energy_cycle_hz; col="#DB4437", fill90="rgba(219,68,55,0.15)", fill68="rgba(219,68,55,0.30)", name="HZ energy-price cycle")...
        ], base_layout("Cost-push cycle and HZ energy-price cycle"))
        save_and_display(cp_ep_fig, "cost_push_energy_price_comparison.png")

        println("Correlation between cost-push and HZ energy-price cycle: ", finite_cor(cost_push_cpi.med, energy_cycle_hz.med))
        println("Relative standard deviation, cost-push / HZ energy-price cycle: ", finite_rel_vol(cost_push_cpi.med, energy_cycle_hz.med))
        println("Relative standard deviation, HZ energy-price cycle / cost-push: ", finite_rel_vol(energy_cycle_hz.med, cost_push_cpi.med))
    end
end

nothing


┌ Warning: saved type BoolParSsm{BitVector, BitMatrix} is missing field Q_cov in workspace type; reconstructing
└ @ JLD2 C:\Users\wrc938\.julia\packages\JLD2\SgtOb\src\data\reconstructing_datatypes.jl:197
┌ Warning: saved type SizeParSsm{Int64} is missing field Q_cov in workspace type; reconstructing
└ @ JLD2 C:\Users\wrc938\.julia\packages\JLD2\SgtOb\src\data\reconstructing_datatypes.jl:197


Saved figure: results_two_gap_iis\cost_push_energy_price_comparison.png
Correlation between cost-push and HZ energy-price cycle: 0.9447442942078269
Relative standard deviation, cost-push / HZ energy-price cycle: 0.9887575852192954
Relative standard deviation, HZ energy-price cycle / cost-push: 1.011370243777408
